# Text-to-Image FLUX 🗒️🖼️

⚠️ **Remember to copy this notebook in your Drive and rename.**

🤗 This notebook uses [Hugging Face Diffusers](https://huggingface.co/docs/diffusers/en/index) to create pipelines for tasks such as image generation.

*Workflows for IAAC MaCDA GenAI  (Apr - Jun 2026) taught by [James McBennett](https://www.linkedin.com/in/mcbennett/) and [Aymeric Brouez](https://www.linkedin.com/in/aymeric-brouez/)*

*With special thanks to past faculty [Nono Martínez Alonso](https://youtube.com/NonoMartinezAlonso).*

## Mount Drive

In [ ]:
# Mount Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

## Hugging Face Token

In [ ]:
# Sign up at Hugging Face and create a "Read" access token (not the default "Fine-Grained" token).
# Click the 🔑 "Secrets" icon in the left sidebar.
# Enable Notebook Access, Set the Name to "HF_TOKEN", Paste your token as the Value

from google.colab import userdata
hf_token = userdata.get("HF_TOKEN")

## Setup

In [ ]:
%cd /content
!rm -rf iaac_genai
!git clone https://github.com/jamesmcbennett/iaac_genai
%cd /content/iaac_genai/

In [ ]:
import sys
sys.path.append('/content/iaac_genai')

In [ ]:
!pip install -q -r requirements.txt --quiet > /dev/null 2>&1

In [ ]:
from config import Config
from utils import set_image_path, save_image, save_yml, save_svg
import torch

import warnings
warnings.filterwarnings("ignore")

## Load pipeline

Load a pipeline with Hugging Face Diffusers.

In [ ]:
# Create pipeline (2 min load on A100)
from diffusers import FluxPipeline
pipe = FluxPipeline.from_pretrained("black-forest-labs/FLUX.1-schnell", torch_dtype=torch.bfloat16, token=hf_token)
pipe.enable_model_cpu_offload()

## Config

You can override parameters here.

In [ ]:
Config.OUTPUT_DIR = '/content/drive/MyDrive/iaac_genai/outputs'

Config.PROMPT = 'a scenic wide-angle landscape of a tall mountain, in the distance is a modern prestressed concrete house designed by Tadao Ando, a lake, and pink cherry blossoms trees'
Config.SEED = 7797676568
Config.STEPS = 28

Config.AUTHOR = 'James McBennett'
Config.ALGO_TYPE = 'Text to Image'
Config.ALGO_NAME = 'Flux.1 schnell'

Config.check()

## Generate

In [ ]:
# Generate.
generator = torch.Generator(Config.TORCH_DEVICE).manual_seed(Config.SEED)
image = pipe(Config.PROMPT, height=1024, width=1024, num_inference_steps=Config.STEPS, max_sequence_length=256, generator=generator).images[0]

display(image)
set_image_path()

## Save

Save the pipeline and config metadata, generated image, and the parameters image.

In [ ]:
# Save image
save_image(image)

In [ ]:
# Save yml metadata
save_yml(pipe)

In [ ]:
# Save svg parameters image
save_svg({
    'SEED': Config.SEED,
    'STEPS': Config.STEPS,
    'Google Colab': '',
})

## Disconnect

In [ ]:
from google.colab import runtime
runtime.unassign()